# Machine 4 of 4: dataset `pubmedqa` + run `full_s0`

The Qwen fine-tuning chain (GSM8K → WikiSQL → XSum → PubMedQA) cannot be split by dataset, so the 4 machines share the work like this:

| Machine | Dataset (S2L selection) | Fine-tuning run |
|---|---|---|
| 1 | gsm8k | `none` |
| 2 | wikisql | `random` 0.02 |
| 3 | xsum | `s2l` 0.02 |
| 4 | pubmedqa (data only) | `full` |

**Phase 1 (dataset `pubmedqa`):** only caches its data. PubMedQA is the last task, so nothing replays it and no S2L buffer is needed.
**Phase 2:** wait until all 4 datasets are cached, then run **full replay (upper bound, the longest run)** and push the result.

Machines exchange files through a private Hugging Face dataset repo (`S2L_REPO`), using your `HF_TOKEN`.

**Before you start**
1. Create a Hugging Face token with **write** permission. In Colab, open the key icon (Secrets) on the left, add `HF_TOKEN`, and turn on *Notebook access*.
2. Pick a repo name such as `YOUR_USERNAME/s2l-exchange` and put the same name in the setup cell on all 4 machines.
3. Runtime → GPU. Start all 4 notebooks at about the same time; the `wait` cells block until what they need exists.
4. After the `pip install` cell, restart the session and continue from the setup cell.

In [ ]:
!nvidia-smi
import torch; print('cuda:', torch.cuda.is_available(), '| bf16:', torch.cuda.is_available() and torch.cuda.is_bf16_supported())

## 1. Setup

In [ ]:
# Setup: Drive folder + Hugging Face token. Re-run this cell after every restart/reconnect.
from google.colab import drive
import os
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/s2l_machine4'
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    import getpass
    os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face token (write access): ')
os.environ['S2L_REPO'] = 'YOUR_HF_USERNAME/s2l-exchange'   # <-- EDIT; identical on all 4 machines
assert 'YOUR_HF_USERNAME' not in os.environ['S2L_REPO'], 'Edit S2L_REPO first'
print('working in', os.getcwd(), '| repo:', os.environ['S2L_REPO'])

## 2. Write the project files (overwrites files with the same name in the folder)

In [ ]:
%%writefile common.py
"""
Shared pieces used by every script: config, data loading, prompt format, encoding, metrics.

Tasks (in training order): GSM8K -> WikiSQL -> XSum -> PubMedQA
Target model: Qwen2.5-0.5B (LoRA).  Proxy model for S2L: Pythia-160M.

IMPORTANT: the first call to load_task() caches the selected rows to data/<task>.json.
Buffer files store INDICES into those cached rows, so every teammate must use the same
data/ folder (commit it to git) or the indices will point at the wrong examples.
"""
import json
import os
import re

TARGET_MODEL = "Qwen/Qwen2.5-0.5B"
PROXY_MODEL = "EleutherAI/pythia-160m"
SEED = 0                    # fixes the data subset; run seeds only change training randomness
DATA_DIR = "data"
ORDER = ["gsm8k", "wikisql", "xsum", "pubmedqa"]
MAX_CHARS = 1500            # truncate long inputs (XSum articles, PubMedQA abstracts)

TASKS = {
    "gsm8k":    dict(n_train=7000, n_test=300, max_new=256),   # full GSM8K train is ~7.47k
    "wikisql":  dict(n_train=8000, n_test=300, max_new=96),
    "xsum":     dict(n_train=8000, n_test=300, max_new=64),
    # PubMedQA's labelled set has only 1,000 examples (yes/no/maybe): split 700 train / 300 test
    "pubmedqa": dict(n_train=700,  n_test=300, max_new=6),
}


def root(debug):
    """Debug runs write to ./debug so they never overwrite real results or buffers."""
    return "debug" if debug else "."


# ------------------------------------------------------------------ formatting
def _fmt_gsm8k(x):
    return ("Solve the math problem step by step. End with '#### <answer>'.\n\n" + x["question"],
            x["answer"])


def _fmt_wikisql(x):
    t = x["table"]
    if isinstance(t, str):
        t = json.loads(t)
    s = x["sql"]
    if isinstance(s, str):
        s = json.loads(s)
    cols = " | ".join(t["header"])
    return (f"Write a SQL query that answers the question.\n\nTable columns: {cols}\n\n"
            f"Question: {x['question']}", s["human_readable"])


def _fmt_xsum(x):
    return ("Summarize the article in one sentence.\n\n" + x["document"][:MAX_CHARS],
            x["summary"].strip())


def _fmt_pubmedqa(x):
    ctx = x["context"]
    if isinstance(ctx, dict):
        ctx = ctx["contexts"]
    abstract = " ".join(ctx)[:MAX_CHARS]
    return (f"Answer the question with yes, no, or maybe, using the abstract.\n\n"
            f"Abstract: {abstract}\n\nQuestion: {x['question']}", x["final_decision"])


FMT = {"gsm8k": _fmt_gsm8k, "wikisql": _fmt_wikisql, "xsum": _fmt_xsum, "pubmedqa": _fmt_pubmedqa}


# ------------------------------------------------------------------ loading
def _load_wikisql():
    from datasets import load_dataset
    errors = []
    # WikiSQL on the Hub uses a loading script. That needs datasets<4 and trust_remote_code.
    try:
        return load_dataset("Salesforce/wikisql", trust_remote_code=True)
    except Exception as e:  # noqa: BLE001
        errors.append(f"script load: {e}")
    # Fallback: the auto-converted parquet branch (not verified for this dataset's schema)
    try:
        return load_dataset("Salesforce/wikisql", revision="refs/convert/parquet")
    except Exception as e:  # noqa: BLE001
        errors.append(f"parquet branch: {e}")
    raise RuntimeError(
        "Could not load Salesforce/wikisql. Try: pip install 'datasets>=2.19,<4.0' "
        "(script-based datasets are not supported in datasets>=4).\n" + "\n".join(errors))


def _load_raw(name):
    from datasets import load_dataset
    if name == "gsm8k":
        ds = load_dataset("openai/gsm8k", "main")
        return ds["train"], ds["test"]
    if name == "wikisql":
        ds = _load_wikisql()
        return ds["train"], ds["test"]
    if name == "xsum":
        ds = load_dataset("EdinburghNLP/xsum")
        return ds["train"], ds["test"]
    if name == "pubmedqa":
        ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled")["train"]  # 1,000 rows, one split
        sp = ds.train_test_split(test_size=TASKS[name]["n_test"], seed=SEED)
        return sp["train"], sp["test"]
    raise ValueError(name)


def load_task(name, debug=False):
    """Returns (train_rows, test_rows); each row is {"prompt": str, "response": str}."""
    path = os.path.join(DATA_DIR, f"{name}.json")
    if os.path.exists(path):
        with open(path) as f:
            d = json.load(f)
    else:
        cfg = TASKS[name]
        tr, te = _load_raw(name)
        tr = tr.shuffle(seed=SEED).select(range(min(cfg["n_train"], len(tr))))
        te = te.shuffle(seed=SEED).select(range(min(cfg["n_test"], len(te))))

        def to_rows(ds):
            return [dict(zip(("prompt", "response"), FMT[name](x))) for x in ds]

        d = {"train": to_rows(tr), "test": to_rows(te)}
        os.makedirs(DATA_DIR, exist_ok=True)
        with open(path, "w") as f:
            json.dump(d, f)
    if debug:  # tiny slice for pipeline smoke tests
        d = {"train": d["train"][:50], "test": d["test"][:20]}
    return d["train"], d["test"]


# ------------------------------------------------------------------ prompt + encoding
def build_prompt(p):
    return f"### Instruction:\n{p}\n\n### Response:\n"


def encode(tok, row, maxlen):
    """Token ids + labels. Loss is computed on the response only (prompt tokens are -100)."""
    p = tok(build_prompt(row["prompt"]), add_special_tokens=False).input_ids
    r = tok(row["response"], add_special_tokens=False).input_ids + [tok.eos_token_id]
    return {"input_ids": (p + r)[:maxlen], "labels": ([-100] * len(p) + r)[:maxlen]}


def collate(batch, pad_id):
    import torch
    m = max(len(x["input_ids"]) for x in batch)
    ids = torch.tensor([x["input_ids"] + [pad_id] * (m - len(x["input_ids"])) for x in batch])
    lab = torch.tensor([x["labels"] + [-100] * (m - len(x["labels"])) for x in batch])
    att = torch.tensor([[1] * len(x["input_ids"]) + [0] * (m - len(x["input_ids"])) for x in batch])
    return {"input_ids": ids, "labels": lab, "attention_mask": att}


# ------------------------------------------------------------------ metrics
def clean(pred):
    # Cut off if the model starts a new "### Instruction:" block. Must not match the "####"
    # answer marker used by GSM8K, so require exactly three '#' followed by whitespace.
    return re.split(r"\n###\s", "\n" + pred.strip())[0].strip()


def gsm_ans(s):
    m = re.findall(r"####\s*\$?\s*(-?[\d,]*\.?\d+)", s)
    if not m:
        return None
    try:
        return float(m[-1].replace(",", ""))
    except ValueError:
        return None


def _norm_sql(s):
    return re.sub(r"\s+", " ", s.lower().replace('"', "'").strip().rstrip(";")).strip()


def score(task, preds, refs):
    """gsm8k/wikisql/pubmedqa -> accuracy; xsum -> ROUGE-L F1. All in [0, 1]."""
    preds = [clean(p) for p in preds]
    n = len(refs)
    if task == "gsm8k":
        ok = 0
        for p, r in zip(preds, refs):
            a, b = gsm_ans(p), gsm_ans(r)
            ok += a is not None and b is not None and abs(a - b) < 1e-6
        return ok / n
    if task == "wikisql":
        return sum(_norm_sql(p) == _norm_sql(r) for p, r in zip(preds, refs)) / n
    if task == "xsum":
        from rouge_score import rouge_scorer
        sc = rouge_scorer.RougeScorer(["rougeL"])
        return sum(sc.score(r, p)["rougeL"].fmeasure for p, r in zip(preds, refs)) / n
    if task == "pubmedqa":
        def first(s):
            w = re.findall(r"[a-z]+", s.lower())
            return w[0] if w else ""
        return sum(first(p) == first(r) for p, r in zip(preds, refs)) / n
    raise ValueError(task)

In [ ]:
%%writefile select_s2l.py
"""
Stage 1 - S2L subset selection with a SMALL PROXY model (Pythia-160M).

For each task, independently:
  1. Fine-tune a fresh proxy on the task's training rows.
  2. At several checkpoints during training, record the loss of EVERY training example
     -> each example gets a "loss trajectory" vector.
  3. Cluster the trajectories with k-means.
  4. Sample the same number of examples from each cluster (small clusters are taken whole,
     leftover budget goes to larger clusters) -> the replay buffer.
  5. Save the chosen indices to buffers/s2l_<ratio>/<task>.json

The Qwen target model is NOT used here.

Usage:
  python select_s2l.py --ratios 0.02,0.05   # ~140-160 / ~350-400 rows per task
  python select_s2l.py --debug            # tiny smoke test, writes to ./debug
  python select_s2l.py --reuse            # reuse saved trajectories, only re-cluster
"""
import argparse
import contextlib
import gc
import json
import os
import random

import numpy as np
import torch
import torch.nn.functional as F

import common as C

MAXLEN_PROXY = 512


def amp_ctx(device, use_amp):
    if use_amp:
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return contextlib.nullcontext()


@torch.no_grad()
def per_example_loss(model, data, pad_id, device, use_amp, bs=8):
    """Mean token loss (response tokens only) for every example, in dataset order."""
    model.eval()
    out = []
    for i in range(0, len(data), bs):
        b = {k: v.to(device) for k, v in C.collate(data[i:i + bs], pad_id).items()}
        with amp_ctx(device, use_amp):
            logits = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).logits
        logits = logits.float()[:, :-1]
        lab = b["labels"][:, 1:]
        tok_loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), lab.reshape(-1),
                                   ignore_index=-100, reduction="none").view(lab.shape)
        mask = (lab != -100).float()
        out += ((tok_loss * mask).sum(1) / mask.sum(1).clamp(min=1)).tolist()
    model.train()
    return out


def train_and_record(model, tok, rows, epochs, bs, lr, n_ckpts, device, use_amp, seed):
    """Fine-tune the proxy; return an (N, n_ckpts) array of per-example losses."""
    data = [C.encode(tok, r, MAXLEN_PROXY) for r in rows]
    n = len(data)
    total = epochs * ((n + bs - 1) // bs)
    ckpt_steps = set(np.unique(np.linspace(total / n_ckpts, total, n_ckpts).round().astype(int)).tolist())
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    rng = random.Random(seed)
    traj, step = [], 0
    model.train()
    for ep in range(epochs):
        order = list(range(n))
        rng.shuffle(order)
        for i in range(0, n, bs):
            b = {k: v.to(device) for k, v in C.collate([data[j] for j in order[i:i + bs]], tok.pad_token_id).items()}
            with amp_ctx(device, use_amp):
                loss = model(**b).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            opt.zero_grad(set_to_none=True)
            step += 1
            if step in ckpt_steps:
                traj.append(per_example_loss(model, data, tok.pad_token_id, device, use_amp))
                print(f"    checkpoint {len(traj)}/{len(ckpt_steps)} at step {step}/{total} "
                      f"(train loss {loss.item():.3f}, mean example loss {np.mean(traj[-1]):.3f})")
    return np.array(traj).T


def balanced_sample(labels, m, rng):
    """Pick m indices, spreading the budget evenly across clusters (small clusters taken whole)."""
    clusters = {c: [int(i) for i in np.where(labels == c)[0]] for c in sorted(set(labels.tolist()))}
    for c in clusters:
        rng.shuffle(clusters[c])
    order = sorted(clusters, key=lambda c: len(clusters[c]))
    chosen, remaining = [], m
    for i, c in enumerate(order):
        take = min(len(clusters[c]), remaining // (len(order) - i))
        chosen += clusters[c][:take]
        remaining -= take
    if remaining > 0:  # only if integer division left a remainder
        left = [i for i in range(len(labels)) if i not in set(chosen)]
        chosen += rng.sample(left, remaining)
    return sorted(chosen)


def select(traj, ratio, k, seed):
    from sklearn.cluster import KMeans
    n = len(traj)
    m = max(1, int(round(ratio * n)))
    k = max(1, min(k, m))
    labels = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(traj)
    return balanced_sample(labels, m, random.Random(seed)), labels


def plot_clusters(traj, labels, path, title):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return
    plt.figure(figsize=(5, 3.5))
    for c in sorted(set(labels.tolist())):
        plt.plot(traj[labels == c].mean(0), alpha=0.8)
    plt.xlabel("checkpoint")
    plt.ylabel("mean example loss")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=120)
    plt.close()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--tasks", default=",".join(C.ORDER[:-1]),
                    help="default skips the LAST task: nothing replays it, so it needs no buffer")
    ap.add_argument("--ratios", default="0.02")
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--bs", type=int, default=8)
    ap.add_argument("--lr", type=float, default=5e-5)
    ap.add_argument("--ckpts", type=int, default=8)
    ap.add_argument("--k", type=int, default=50, help="number of k-means clusters (capped at buffer size)")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--reuse", action="store_true", help="reuse saved trajectories")
    ap.add_argument("--debug", action="store_true")
    args = ap.parse_args()

    R = C.root(args.debug)
    ratios = [float(x) for x in args.ratios.split(",")]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    use_amp = device == "cuda" and torch.cuda.is_bf16_supported()
    os.makedirs(f"{R}/trajectories", exist_ok=True)
    os.makedirs(f"{R}/diagnostics", exist_ok=True)

    for t in args.tasks.split(","):
        print(f"\n=== {t} ===")
        rows, _ = C.load_task(t, debug=args.debug)
        tpath = f"{R}/trajectories/{t}.npy"
        if args.reuse and os.path.exists(tpath):
            traj = np.load(tpath)
            print(f"  loaded trajectories {traj.shape}")
        else:
            from transformers import AutoModelForCausalLM, AutoTokenizer
            tok = AutoTokenizer.from_pretrained(C.PROXY_MODEL)
            if tok.pad_token is None:
                tok.pad_token = tok.eos_token
            model = AutoModelForCausalLM.from_pretrained(C.PROXY_MODEL, torch_dtype=torch.float32).to(device)
            traj = train_and_record(model, tok, rows, args.epochs, args.bs, args.lr,
                                    args.ckpts, device, use_amp, args.seed)
            np.save(tpath, traj)
            del model
            gc.collect()
            if device == "cuda":
                torch.cuda.empty_cache()
        print(f"  trajectories {traj.shape}; mean loss first->last checkpoint: "
              f"{traj[:, 0].mean():.3f} -> {traj[:, -1].mean():.3f}")
        if np.abs(traj[:, 0].mean() - traj[:, -1].mean()) < 0.05:
            print("  WARNING: trajectories are nearly flat - the proxy barely learned this task; "
                  "clusters may carry little information.")

        for r in ratios:
            idx, labels = select(traj, r, args.k, args.seed)
            out_dir = f"{R}/buffers/s2l_{r}"
            os.makedirs(out_dir, exist_ok=True)
            with open(f"{out_dir}/{t}.json", "w") as f:
                json.dump(idx, f)
            sizes = np.bincount(labels).tolist()
            print(f"  ratio {r}: selected {len(idx)}/{len(rows)} examples; cluster sizes {sizes}")
            plot_clusters(traj, labels, f"{R}/diagnostics/{t}_{r}.png", f"{t} (ratio {r})")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile sequential_finetune.py
"""
Stage 2 - Sequential (continual) fine-tuning of Qwen2.5-0.5B with LoRA.

One LoRA adapter is carried through all tasks:
    stage 1: D1
    stage 2: D2 + buffer(1)
    stage 3: D3 + buffer(1) + buffer(2)
    stage 4: D4 + buffer(1) + buffer(2) + buffer(3)
After every stage the model is evaluated on ALL tasks -> results matrix R[after_task][eval_task].

Replay modes:
    none    no replay (lower bound, shows forgetting)
    random  random subset of each earlier task, same size as the S2L buffer
    s2l     buffers/s2l_<ratio>/<task>.json produced by select_s2l.py
    full    ALL data of earlier tasks (upper bound)

Usage:
    python sequential_finetune.py --debug --replay none      # smoke test first!
    python sequential_finetune.py --replay none
    python sequential_finetune.py --replay random --ratio 0.02
    python sequential_finetune.py --replay s2l --ratio 0.02
"""
import argparse
import json
import os
import random
import time

import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

import common as C

MAXLEN = 768


class SFTData(torch.utils.data.Dataset):
    def __init__(self, rows, tok):
        self.items = [C.encode(tok, r, MAXLEN) for r in rows]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


@torch.no_grad()
def generate(model, tok, prompts, max_new, bs=16):
    model.eval()
    tok.padding_side = "left"
    outs = []
    for i in range(0, len(prompts), bs):
        batch = [C.build_prompt(p) for p in prompts[i:i + bs]]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in outs]


def eval_all(model, tok, tests):
    res = {}
    for t, rows in tests.items():
        preds = generate(model, tok, [r["prompt"] for r in rows], C.TASKS[t]["max_new"])
        res[t] = C.score(t, preds, [r["response"] for r in rows])
        print(f"    eval {t}: {res[t]:.3f}   e.g. pred={preds[0][:80]!r}")
    return res


def make_buffer(mode, task, rows, ratio, seed, root):
    if mode == "none":
        return []
    if mode == "full":
        return list(rows)
    if mode == "random":
        k = max(1, int(round(ratio * len(rows))))
        return random.Random(seed).sample(rows, k)
    if mode == "s2l":
        with open(f"{root}/buffers/s2l_{ratio}/{task}.json") as f:
            idx = json.load(f)
        assert max(idx) < len(rows), f"buffer index out of range for {task}: data/ and buffers/ are out of sync"
        return [rows[i] for i in idx]
    raise ValueError(mode)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--order", default=",".join(C.ORDER))
    ap.add_argument("--replay", choices=["none", "random", "s2l", "full"], default="none")
    ap.add_argument("--ratio", type=float, default=0.02)
    ap.add_argument("--epochs", type=float, default=2)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--debug", action="store_true", help="50 train / 20 test examples, 1 epoch")
    args = ap.parse_args()

    order = args.order.split(",")
    R_ = C.root(args.debug)
    name = f"{args.replay}_s{args.seed}" if args.replay in ("none", "full") else f"{args.replay}_{args.ratio}_s{args.seed}"
    os.makedirs(f"{R_}/results/{name}", exist_ok=True)
    random.seed(args.seed)
    torch.manual_seed(args.seed)

    tok = AutoTokenizer.from_pretrained(C.TARGET_MODEL)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    model = AutoModelForCausalLM.from_pretrained(
        C.TARGET_MODEL, torch_dtype=torch.bfloat16 if use_bf16 else torch.float16, device_map="auto")
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    if not use_bf16:  # fp16 GPUs (e.g. Colab T4): keep trainable LoRA weights in fp32
        for p in model.parameters():
            if p.requires_grad:
                p.data = p.data.float()
    model.print_trainable_parameters()

    train, test = {}, {}
    for t in order:
        train[t], test[t] = C.load_task(t, debug=args.debug)
        print(f"{t}: {len(train[t])} train / {len(test[t])} test")

    # LoRA B matrices start at zero, so this equals the zero-shot base model
    print("\n=== zero-shot base model ===")
    R = {"base": eval_all(model, tok, test)}

    for k, t in enumerate(order):
        rows = list(train[t])
        for prev in order[:k]:
            rows += make_buffer(args.replay, prev, train[prev], args.ratio, args.seed, R_)
        random.Random(args.seed).shuffle(rows)
        print(f"\n=== Stage {k + 1}/{len(order)}: {t} | {len(train[t])} new + {len(rows) - len(train[t])} replay rows ===")

        t0 = time.time()
        targs = TrainingArguments(
            output_dir=f"{R_}/tmp/{name}", per_device_train_batch_size=8, gradient_accumulation_steps=2,
            learning_rate=args.lr, num_train_epochs=1 if args.debug else args.epochs,
            lr_scheduler_type="cosine", warmup_ratio=0.03, logging_steps=20, save_strategy="no",
            report_to="none", bf16=use_bf16, fp16=not use_bf16, remove_unused_columns=False, seed=args.seed)
        Trainer(model=model, args=targs, train_dataset=SFTData(rows, tok),
                data_collator=lambda b: C.collate(b, tok.pad_token_id)).train()
        print(f"  trained in {(time.time() - t0) / 60:.1f} min")

        model.save_pretrained(f"{R_}/adapters/{name}/after_{t}")
        R[t] = eval_all(model, tok, test)
        with open(f"{R_}/results/{name}/results.json", "w") as f:
            json.dump({"order": order, "replay": args.replay, "ratio": args.ratio,
                       "seed": args.seed, "R": R}, f, indent=2)

    print("\nResults matrix (rows = after training on, cols = evaluated on)")
    print("after".ljust(10) + "".join(c.ljust(10) for c in order))
    for r, v in R.items():
        print(r.ljust(10) + "".join(f"{v[c]:.3f}".ljust(10) for c in order))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile aggregate_results.py
"""
Stage 3 - Aggregate results/<run>/results.json into tables and a plot.

Runs named like  s2l_0.05_s0, s2l_0.05_s1  are grouped (mean +- std over seeds) as  s2l_0.05.

Metrics (order = [T1..Tn], R[a][b] = score on task b after training through task a):
  avg final  = mean_j R[Tn][Tj]
  forgetting = mean over j<n of ( max_{j<=l<=n-1} R[Tl][Tj]  -  R[Tn][Tj] )   (lower is better)
  BWT        = mean over j<n of ( R[Tn][Tj] - R[Tj][Tj] )                     (higher is better)

Usage:  python aggregate_results.py [--root .]
"""
import argparse
import csv
import glob
import json
import os
import re
from collections import defaultdict

import numpy as np


def metrics(R, order):
    T = len(order)
    last = R[order[-1]]
    avg = float(np.mean([last[t] for t in order]))
    fgt = float(np.mean([max(R[order[l]][order[j]] for l in range(j, T - 1)) - last[order[j]]
                         for j in range(T - 1)]))
    bwt = float(np.mean([last[order[j]] - R[order[j]][order[j]] for j in range(T - 1)]))
    return avg, fgt, bwt


def ms(xs):
    return f"{np.mean(xs):.3f} ± {np.std(xs):.3f}" if len(xs) > 1 else f"{xs[0]:.3f}"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default=".")
    args = ap.parse_args()

    groups, order, base = defaultdict(list), None, None
    for path in sorted(glob.glob(f"{args.root}/results/*/results.json")):
        d = json.load(open(path))
        if order is None:
            order, base = d["order"], d["R"]["base"]
        if d["order"] != order or d["order"][-1] not in d["R"]:
            print(f"skipping {path} (different task order or unfinished run)")
            continue
        method = re.sub(r"_s\d+$", "", os.path.basename(os.path.dirname(path)))
        groups[method].append(d["R"])
    if not groups:
        print("no finished runs found")
        return

    rows = []
    print(f"\nTask order: {' -> '.join(order)}   (zero-shot: " +
          ", ".join(f"{t} {base[t]:.3f}" for t in order) + ")\n")
    header = ["method", "seeds", "avg_final", "forgetting", "BWT"] + [f"final_{t}" for t in order]
    print("| " + " | ".join(header) + " |")
    print("|" + "---|" * len(header))
    for m, runs in groups.items():
        mt = np.array([metrics(R, order) for R in runs])
        finals = {t: [R[order[-1]][t] for R in runs] for t in order}
        row = [m, str(len(runs)), ms(mt[:, 0]), ms(mt[:, 1]), ms(mt[:, 2])] + [ms(finals[t]) for t in order]
        rows.append(row)
        print("| " + " | ".join(row) + " |")

    os.makedirs(f"{args.root}/results", exist_ok=True)
    with open(f"{args.root}/results/summary.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return
    methods = list(groups)
    width = 0.8 / (len(methods) + 1)
    x = np.arange(len(order))
    plt.figure(figsize=(8, 4))
    plt.bar(x - 0.4 + width / 2, [base[t] for t in order], width, label="zero-shot", color="lightgray")
    for i, m in enumerate(methods):
        means = [np.mean([R[order[-1]][t] for R in groups[m]]) for t in order]
        errs = [np.std([R[order[-1]][t] for R in groups[m]]) for t in order]
        plt.bar(x - 0.4 + width * (i + 1.5), means, width, yerr=errs, label=m)
    plt.xticks(x, order)
    plt.ylabel("score after the final stage")
    plt.title("Performance on each task after training on all tasks")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(f"{args.root}/results/final_scores.png", dpi=150)
    print(f"\nsaved {args.root}/results/summary.csv and final_scores.png")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile prepare_data.py
"""
Download + cache selected rows for the given tasks into data/<task>.json (same code path as every
other script, so the rows are identical). Used by the multi-machine notebooks.

Usage: python prepare_data.py --tasks gsm8k
"""
import argparse

import common as C

ap = argparse.ArgumentParser()
ap.add_argument("--tasks", default=",".join(C.ORDER))
for t in ap.parse_args().tasks.split(","):
    tr, te = C.load_task(t)
    print(f"{t}: cached {len(tr)} train / {len(te)} test rows -> data/{t}.json")

In [ ]:
%%writefile sync.py
"""
Exchange files between machines through a PRIVATE Hugging Face dataset repo.

Environment (set in the notebook):
  HF_TOKEN  - Hugging Face token with WRITE permission
  S2L_REPO  - 'username/repo-name' (same on every machine; created automatically, private)

Usage (paths are relative to the project folder; a path can be a file or a folder):
  python sync.py push data/gsm8k.json buffers trajectories
  python sync.py pull data buffers
  python sync.py wait data/gsm8k.json data/xsum.json [--timeout 12] [--poll 60]
      -> blocks until every listed file exists in the repo (timeout in hours)
"""
import argparse
import os
import sys
import time

from huggingface_hub import HfApi, snapshot_download


def _cfg():
    repo, tok = os.environ.get("S2L_REPO", ""), os.environ.get("HF_TOKEN", "")
    if "/" not in repo:
        sys.exit("Set S2L_REPO='username/repo-name' (same on every machine).")
    if not tok:
        sys.exit("Set HF_TOKEN (needs WRITE permission).")
    return repo, tok, HfApi(token=tok)


def _norm(p):
    return os.path.normpath(p).replace("\\", "/")


def _retry(fn, tries=4):
    for i in range(tries):
        try:
            return fn()
        except Exception as e:  # noqa: BLE001
            if i == tries - 1:
                raise
            print(f"  retry {i + 1}/{tries - 1}: {e}")
            time.sleep(5 * (i + 1))


def push(paths):
    repo, _, api = _cfg()
    _retry(lambda: api.create_repo(repo, repo_type="dataset", private=True, exist_ok=True))
    for p in map(_norm, paths):
        if os.path.isdir(p):
            _retry(lambda p=p: api.upload_folder(folder_path=p, path_in_repo=p, repo_id=repo,
                                                 repo_type="dataset", commit_message=f"push {p}"))
        elif os.path.isfile(p):
            _retry(lambda p=p: api.upload_file(path_or_fileobj=p, path_in_repo=p, repo_id=repo,
                                               repo_type="dataset", commit_message=f"push {p}"))
        else:
            print(f"  skip (not found): {p}")
            continue
        print(f"  pushed {p}")


def pull(paths):
    repo, tok, _ = _cfg()
    pats = []
    for p in map(_norm, paths):
        pats += [p, p + "/*"]  # file or whole folder
    _retry(lambda: snapshot_download(repo_id=repo, repo_type="dataset", allow_patterns=pats,
                                     local_dir=".", token=tok))
    print(f"  pulled {', '.join(map(_norm, paths))}")


def wait(files, timeout_h, poll):
    repo, _, api = _cfg()
    files = [_norm(f) for f in files]
    end, last = time.time() + timeout_h * 3600, None
    while True:
        try:
            have = set(api.list_repo_files(repo, repo_type="dataset"))
        except Exception as e:  # repo may not exist yet
            have = set()
            print(f"  repo not readable yet ({type(e).__name__})")
        missing = [f for f in files if f not in have]
        if not missing:
            print("  all files present")
            return
        if time.time() > end:
            sys.exit(f"timed out waiting for: {missing}")
        if missing != last:
            print(f"{time.strftime('%H:%M:%S')} waiting for {missing}")
            last = missing
        time.sleep(poll)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    for c in ("push", "pull"):
        sub.add_parser(c).add_argument("paths", nargs="+")
    w = sub.add_parser("wait")
    w.add_argument("files", nargs="+")
    w.add_argument("--timeout", type=float, default=12.0, help="hours")
    w.add_argument("--poll", type=int, default=60, help="seconds between checks")
    a = ap.parse_args()
    {"push": lambda: push(a.paths), "pull": lambda: pull(a.paths),
     "wait": lambda: wait(a.files, a.timeout, a.poll)}[a.cmd]()

In [ ]:
%%writefile requirements.txt
torch
transformers
peft
accelerate
# WikiSQL on the Hub is script-based, which needs datasets<4 (and trust_remote_code)
datasets>=2.19,<4.0
scikit-learn
rouge_score
matplotlib
numpy

## 3. Install requirements, then restart the session
This pins `datasets<4` (needed for WikiSQL).

In [ ]:
!pip install -q -r requirements.txt

### ⟳ Runtime → Restart session, then run the setup cell again (section 4).

## 4. Re-run setup after the restart

In [ ]:
# Setup: Drive folder + Hugging Face token. Re-run this cell after every restart/reconnect.
from google.colab import drive
import os
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/s2l_machine4'
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    import getpass
    os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face token (write access): ')
os.environ['S2L_REPO'] = 'YOUR_HF_USERNAME/s2l-exchange'   # <-- EDIT; identical on all 4 machines
assert 'YOUR_HF_USERNAME' not in os.environ['S2L_REPO'], 'Edit S2L_REPO first'
print('working in', os.getcwd(), '| repo:', os.environ['S2L_REPO'])
import datasets; print('datasets', datasets.__version__, '(should be < 4.0)')

## 5. Phase 1: cache `pubmedqa` and share it
Other machines need this file for evaluation, so push it right away.

In [ ]:
!python prepare_data.py --tasks pubmedqa
!python sync.py push data/pubmedqa.json

## 6. Phase 1: no selection needed for this machine

## 7. Phase 2: wait for all 4 datasets, then pull them

In [ ]:
!python sync.py wait data/gsm8k.json data/wikisql.json data/xsum.json data/pubmedqa.json
!python sync.py pull data
!ls data

## 8. Fine-tuning run: full replay (upper bound, the longest run)
4 stages × 2 epochs, evaluated on all tasks after each stage. Long step. Results go to `results/full_s0/`.

In [ ]:
!python sequential_finetune.py --replay full

## 9. Push the finished result
Only pushed after the run completes, so its presence in the repo means it is finished.

In [ ]:
!python sync.py push results/full_s0